# Notebook 03 — Grad-CAM Examples

Load a trained model and visualise which image regions drive classification decisions.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.paths import load_config
from src.utils.seed import set_seed
from src.data.dataset import ERCPDataset
from src.data.transforms import get_transforms
from src.models.build_model import build_model
from src.training.evaluate import load_checkpoint
from src.explainability.gradcam import (
    get_target_layer, compute_gradcam, overlay_heatmap, save_gradcam_figure
)

cfg = load_config('../config.yaml')
set_seed(cfg['seed'])
CLASS_NAMES = cfg['data']['class_names']

# --- CONFIGURE THESE ---
CHECKPOINT = '../models/best_model.pth'   # Update to your checkpoint
MODEL_NAME = 'efficientnet_b0'            # Update to match checkpoint
# -----------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
model = build_model(MODEL_NAME, num_classes=4, pretrained=False)
model = load_checkpoint(CHECKPOINT, model, device)
model.eval()

target_layer = get_target_layer(model, MODEL_NAME)
print(f'Target layer: {target_layer.__class__.__name__}')

In [ ]:
test_ds = ERCPDataset(
    cfg['data']['test_dir'],
    transform=get_transforms(cfg['training']['image_size'], 'test'),
    class_names=CLASS_NAMES
)
print(f'Test set: {len(test_ds)} images')

## Generate and display Grad-CAM for one image per class

In [ ]:
import torchvision.transforms as T

inv_norm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# Index by class
by_class = {i: [] for i in range(len(CLASS_NAMES))}
for idx, (_, lbl) in enumerate(test_ds.samples):
    by_class[lbl].append(idx)

fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(12, len(CLASS_NAMES)*4))

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    indices = by_class[cls_idx]
    if not indices:
        print(f'No test images for {cls_name}')
        continue
    
    img_tensor, true_label = test_ds[indices[0]]
    
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(out, dim=1)[0].cpu().numpy()
        pred = int(probs.argmax())
    
    heatmap = compute_gradcam(model, img_tensor, target_layer, pred, device)
    
    orig = inv_norm(img_tensor).permute(1,2,0).numpy()
    orig = np.clip(orig, 0, 1)
    orig_uint8 = (orig * 255).astype(np.uint8)
    overlay = overlay_heatmap(orig_uint8, heatmap)
    
    import cv2
    import numpy as np
    hm_resized = cv2.resize(heatmap, (orig_uint8.shape[1], orig_uint8.shape[0]))
    hm_color = cv2.applyColorMap(np.uint8(255*hm_resized), cv2.COLORMAP_JET)
    hm_rgb = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    
    axes[cls_idx][0].imshow(orig)
    axes[cls_idx][0].set_title(f'{cls_name}\nOriginal', fontsize=10)
    axes[cls_idx][0].axis('off')
    
    axes[cls_idx][1].imshow(hm_rgb)
    axes[cls_idx][1].set_title('Grad-CAM Heatmap', fontsize=10)
    axes[cls_idx][1].axis('off')
    
    match = '✓' if pred == true_label else '✗'
    axes[cls_idx][2].imshow(overlay)
    axes[cls_idx][2].set_title(f'{match} Pred: {CLASS_NAMES[pred]}', fontsize=10)
    axes[cls_idx][2].axis('off')

plt.suptitle(f'Grad-CAM — {MODEL_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/gradcam_overview.png', dpi=150)
plt.show()